# RoadCarEnv 使用样例

演示 `RoadCarEnv` gymnasium 环境的完整用法：
1. 基本用法：创建环境、reset、step 循环
2. 固定道路 vs 随机道路
3. 不同车辆模型配置（运动学/动力学 × 欧拉/RK4）
4. 观测空间解析与可视化
5. 多 episode 运行与奖励曲线

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from sim_env import (
    RoadVehicleEnv, EnvConfig,
    VehicleModelConfig, VehicleParams, ModelType, IntegratorType,
    RoadGenerationConfig, RoadSegmentType, SegmentSpec,
    RewardWeights,
)

plt.rcParams["font.sans-serif"] = ["Arial Unicode MS", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False
%matplotlib inline

## 演示：使用 render() 可视化环境

使用 `render_mode="rgb_array"` 获取 pygame 渲染的帧图像，用 matplotlib 展示。

In [ ]:
# 演示：在不同时刻调用 render() 获取环境状态
env = RoadVehicleEnv(
    EnvConfig(
        road_config=RoadGenerationConfig(
            num_lanes=3,
            fixed_segments=[
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 10}),
                SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 40}),
            ],
        ),
        road_points_ahead=20,
        init_speed=8.0,
        render_mode = "rgb_arr"
    ),
)

snapshots = [10]
frames = []

obs, info = env.reset(seed=0)
frame = env.render()
if frame is not None:
    frames.append((0, frame.copy()))

for step in range(1, max(snapshots) + 1):
    obs, _, _, _, info = env.step(np.array([0.5, 0.00]))
    if step in snapshots:
        frame = env.render(
                      info_text=[f"progress = {info.get("progress", 0.0):6.1%}"],
                      overlays=[
                          {"points": obs["centerline"],   "color": (255, 69, 0), "style": "solid", "width": 2,   "label": "centerline"},
                          {"points": obs["left_boundary"], "color": (255, 69, 0), "style": "solid",   "width": 2,   "label": "left_boundary"},
                          {"points": obs["right_boundary"],"color": (255, 69, 0),   "style": "solid",   "width": 2,   "label": "right_boundary"},
                    ]
                )
        if frame is not None:
            frames.append((step, frame.copy()))
env.close()

if frames:
    n = len(frames)
    fig, axes = plt.subplots(1, n, figsize=(5 * n, 4))
    if n == 1:
        axes = [axes]
    for ax, (s, img) in zip(axes, frames):
        ax.imshow(img)
        ax.set_title(f"step={s}")
        ax.axis("off")
    fig.suptitle("render() 演示：不同时刻的环境状态", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("未能获取渲染帧，需要 pygame: pip install pygame")

In [ ]:
obs.keys(), info.keys()

## 示例 1：基本用法

创建环境 → reset → 循环 step，记录轨迹和奖励。使用简单的恒定加速动作。

In [ ]:
env = RoadVehicleEnv(EnvConfig(
    road_config=RoadGenerationConfig(num_random_segments=5, num_lanes=1),
))

obs, info = env.reset(seed=42)
print("观测空间:")
for k, v in obs.items():
    print(f"  {k:20s}: shape={v.shape}")
print(f"动作空间: {env.action_space}")
print(f"初始状态: x={info['x']:.2f}, y={info['y']:.2f}, v={info['v']:.2f}")

# 运行一个 episode
trajectory = {"x": [], "y": [], "v": [], "reward": []}
total_reward = 0
done = False

while not done:
    action = np.array([1.0, 0.2])
    obs, reward, terminated, truncated, info = env.step(action)
    done = terminated or truncated
    total_reward += reward
    trajectory["x"].append(info["x"])
    trajectory["y"].append(info["y"])
    trajectory["v"].append(info["v"])
    trajectory["reward"].append(reward)

env.close()
print(f"\nEpisode 结束: steps={info['step']}, terminated={terminated}, truncated={truncated}")
print(f"总奖励: {total_reward:.2f}, 最终进度: {info.get('progress', 'N/A')}")

In [ ]:
# 绘制轨迹和奖励曲线
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax = axes[0]
ax.plot(trajectory["x"], trajectory["y"], "b-", lw=1.5)
ax.plot(trajectory["x"][0], trajectory["y"][0], "go", ms=10, label="起点")
ax.plot(trajectory["x"][-1], trajectory["y"][-1], "rs", ms=10, label="终点")
ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("车辆轨迹")
ax.set_aspect("equal")
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(trajectory["v"], "tab:orange")
ax.set_xlabel("步数")
ax.set_ylabel("速度 (m/s)")
ax.set_title("速度曲线")
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.plot(np.cumsum(trajectory["reward"]), "tab:green")
ax.set_xlabel("步数")
ax.set_ylabel("累计奖励")
ax.set_title("累计奖励曲线")
ax.grid(True, alpha=0.3)

fig.suptitle("示例 1：基本用法（恒定加速）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例：render 可视化

使用 `render_mode="rgb_array"` 获取 pygame 渲染的帧图像，用 matplotlib 展示。
每隔若干步抓取一帧，展示车辆在道路上行驶的过程。

In [ ]:
env = RoadVehicleEnv(
    EnvConfig(
        road_config=RoadGenerationConfig(
            num_lanes=2,
            fixed_segments=[
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 60}),
                SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 40}),
            ],
        ),
        init_speed=5.0,
    ),
    render_mode="rgb_array",
)

obs, info = env.reset(seed=0)

# 收集关键帧
frames = []
snapshot_steps = {0, 30, 60, 90, 120, 150}

frame = env.render()
if frame is not None:
    frames.append((0, frame.copy()))

for step in range(1, 300):
    obs, reward, terminated, truncated, info = env.step(np.array([0.5, 0.02]))
    if step in snapshot_steps:
        frame = env.render()
        if frame is not None:
            frames.append((step, frame.copy()))
    if terminated or truncated:
        # 抓最后一帧
        frame = env.render()
        if frame is not None:
            frames.append((step, frame.copy()))
        break

env.close()

if frames:
    n = len(frames)
    fig, axes = plt.subplots(2, (n + 1) // 2, figsize=(5 * ((n + 1) // 2), 8))
    axes = axes.flatten()
    for ax, (s, img) in zip(axes, frames):
        ax.imshow(img)
        ax.set_title(f"step={s}")
        ax.axis("off")
    # 隐藏多余的子图
    for i in range(len(frames), len(axes)):
        axes[i].axis("off")
    fig.suptitle("render 帧序列（rgb_array 模式）", fontsize=14, y=1.02)
    fig.tight_layout()
    plt.show()
else:
    print("未能获取渲染帧，需要 pygame: pip install pygame")

## 示例 2：固定道路 vs 随机道路

对比 `fixed_segments` 指定道路和 `generate_random` 随机道路的效果。

In [ ]:
def run_episode(env, action_fn, max_steps=500):
    """运行一个 episode，返回轨迹。"""
    obs, info = env.reset(seed=42)
    xs, ys, rewards = [info["x"]], [info["y"]], []
    for _ in range(max_steps):
        action = action_fn(obs, info)
        obs, reward, terminated, truncated, info = env.step(action)
        xs.append(info["x"])
        ys.append(info["y"])
        rewards.append(reward)
        if terminated or truncated:
            break
    env.close()
    return np.array(xs), np.array(ys), np.array(rewards)

# 简单动作：轻微加速
simple_action = lambda obs, info: np.array([0.5, 0.2])

# 固定道路
env_fixed = RoadVehicleEnv(EnvConfig(
    road_config=RoadGenerationConfig(
        num_lanes=2,
        fixed_segments=[
            SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 80}),
            SegmentSpec(RoadSegmentType.CURVE, {"radius": 50, "angle_deg": 90}),
            SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 60}),
        ],
    ),
))
fx, fy, fr = run_episode(env_fixed, simple_action)

# 随机道路
env_random = RoadVehicleEnv(EnvConfig(
    road_config=RoadGenerationConfig(num_random_segments=6, num_lanes=2),
))
rx, ry, rr = run_episode(env_random, simple_action)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, xs, ys, title in [
    (axes[0], fx, fy, f"固定道路 (总奖励={fr.sum():.1f})"),
    (axes[1], rx, ry, f"随机道路 (总奖励={rr.sum():.1f})"),
]:
    ax.plot(xs, ys, "b-", lw=1.5)
    ax.plot(xs[0], ys[0], "go", ms=10)
    ax.plot(xs[-1], ys[-1], "rs", ms=10)
    ax.set_title(title)
    ax.set_aspect("equal")
    ax.set_xlabel("x (m)")
    ax.set_ylabel("y (m)")
    ax.grid(True, alpha=0.3)

fig.suptitle("示例 2：固定道路 vs 随机道路", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

## 示例 3：不同车辆模型配置

在同一条固定道路上，对比 4 种模型×积分器组合的轨迹。

In [ ]:
fixed_segs = [
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 60}),
    SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
    SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 40}),
]

configs = [
    (ModelType.KINEMATIC, IntegratorType.EULER, "运动学+欧拉", "tab:blue"),
    (ModelType.KINEMATIC, IntegratorType.RK4,   "运动学+RK4",  "tab:orange"),
    (ModelType.DYNAMIC,   IntegratorType.EULER, "动力学+欧拉", "tab:green"),
    (ModelType.DYNAMIC,   IntegratorType.RK4,   "动力学+RK4",  "tab:red"),
]

fig, ax = plt.subplots(figsize=(12, 8))

for mt, it, label, color in configs:
    env = RoadVehicleEnv(EnvConfig(
        vehicle_config=VehicleModelConfig(
            vehicle_params=VehicleParams(),
            model_type=mt,
            integrator_type=it,
        ),
        road_config=RoadGenerationConfig(
            num_lanes=1,
            fixed_segments=fixed_segs,
        ),
    ))
    xs, ys, rewards = run_episode(env, simple_action)
    ax.plot(xs, ys, color=color, lw=2, label=f"{label} (R={rewards.sum():.0f})")
    ax.plot(xs[0], ys[0], "o", color=color, ms=8)
    ax.plot(xs[-1], ys[-1], "s", color=color, ms=8)

ax.set_xlabel("x (m)")
ax.set_ylabel("y (m)")
ax.set_title("四种模型×积分器组合轨迹对比")
ax.set_aspect("equal")
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 示例 4：观测空间解析

将观测向量拆解为车辆状态和道路点，使用 render() 可视化当前帧的观测内容。

In [ ]:
env = RoadVehicleEnv(
    EnvConfig(
        road_config=RoadGenerationConfig(
            num_lanes=1,
            fixed_segments=[
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 50}),
                SegmentSpec(RoadSegmentType.CURVE, {"radius": 40, "angle_deg": 60}),
                SegmentSpec(RoadSegmentType.STRAIGHT, {"length": 30}),
            ],
        ),
        road_points_ahead=15,
    ),
    render_mode="rgb_array",
)

obs, info = env.reset(seed=0)
# 走几步让车辆移动
for _ in range(30):
    obs, _, _, _, info = env.step(np.array([2.0, 0.05]))

# 获取当前帧
frame = env.render()
env.close()

if frame is not None:
    fig, ax = plt.subplots(figsize=(10, 8))
    ax.imshow(frame)
    ax.set_title("观测解析（使用 render() 可视化）")
    ax.axis("off")
    fig.tight_layout()
    plt.show()
else:
    print("未能获取渲染帧，需要 pygame: pip install pygame")

## 示例 5：多 episode 奖励对比

用不同随机种子跑多个 episode，对比累计奖励分布。

In [ ]:
from sim_env.vehicle_controller import VehicleMPC, MPCConfig

seeds = range(10)
results = {
    "fixed": {"rewards": [], "lengths": [], "progress": []},
    "mpc":   {"rewards": [], "lengths": [], "progress": []},
}

for seed in seeds:
    for policy_name in ["fixed", "mpc"]:
        env = RoadVehicleEnv(EnvConfig(
            road_config=RoadGenerationConfig(num_random_segments=6, num_lanes=1),
            road_points_ahead=25,
            init_speed=5.0,
        ))
        if policy_name == "mpc":
            mpc = VehicleMPC(mpc_config=MPCConfig(vehicle_params=env._vehicle.params, horizon=15, target_speed=5.0, dt=0.1),
            )
            mpc.reset()

        obs, info = env.reset(seed=seed)
        total_r, steps, done = 0, 0, False

        while not done:
            if policy_name == "fixed":
                action = np.array([0.5, 0.0])
            else:
                veh = obs["vehicle"]
                state = np.array([veh[0], veh[1], veh[2], veh[3], info.get("steering", 0.0)])
                action, _, _ = mpc.compute(state, obs["centerline"])

            obs, reward, terminated, truncated, info = env.step(action)
            total_r += reward
            steps += 1
            done = terminated or truncated

        env.close()
        results[policy_name]["rewards"].append(total_r)
        results[policy_name]["lengths"].append(steps)
        results[policy_name]["progress"].append(info.get("progress", 0.0))

# 可视化
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
x = np.arange(len(seeds))
w = 0.35

for ax, key, ylabel, title in [
    (axes[0], "rewards",  "累计奖励", "累计奖励对比"),
    (axes[1], "lengths",  "步数",     "Episode 长度对比"),
    (axes[2], "progress", "进度",     "道路完成进度对比"),
]:
    fixed_vals = results["fixed"][key]
    mpc_vals = results["mpc"][key]
    ax.bar(x - w/2, fixed_vals, w, color="tab:blue", alpha=0.7, label="固定动作")
    ax.bar(x + w/2, mpc_vals, w, color="tab:orange", alpha=0.7, label="MPC")
    ax.axhline(np.mean(fixed_vals), color="tab:blue", ls="--", alpha=0.5)
    ax.axhline(np.mean(mpc_vals), color="tab:orange", ls="--", alpha=0.5)
    ax.set_xlabel("seed")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.set_xticks(x)
    ax.legend()
    ax.grid(True, alpha=0.3)

# progress 图加百分比格式
axes[2].yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f"{v:.0%}"))

fig.suptitle("示例 5：固定动作 vs MPC（10 个随机道路）", fontsize=14, y=1.02)
fig.tight_layout()
plt.show()

print(f"固定动作: 平均奖励={np.mean(results['fixed']['rewards']):.1f}, "
      f"平均步数={np.mean(results['fixed']['lengths']):.0f}, "
      f"平均进度={np.mean(results['fixed']['progress']):.1%}")
print(f"MPC:     平均奖励={np.mean(results['mpc']['rewards']):.1f}, "
      f"平均步数={np.mean(results['mpc']['lengths']):.0f}, "
      f"平均进度={np.mean(results['mpc']['progress']):.1%}")